In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

#! FILE NÀY ĐỂ TẠO RA CODE CSV ~133.63MB VÀ CHUYỂN ĐỊNH DẠNG SANG PARQUET.
# Đã cập nhật: Sinh dữ liệu ảo an toàn tuyệt đối cho K-Means (RFM) và FP-Growth (Association)

# 1. Khởi tạo Spark
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
spark = SparkSession.builder \
    .appName("Safe_Data_Augmentation") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

# Đường dẫn file đầu vào và đầu ra
CSV_INPUT = "SalesTransaction_origin.csv"
OUTPUT_CSV = "../data_augmented.csv"
OUTPUT_PARQUET = "../data.parquet"

# CÁC CỘT QUAN TRỌNG DỰA TRÊN DỮ LIỆU MẪU
TRANSACTION_COL = "TransactionNo"
DATE_COL = "Date"
CUSTOMER_COL = "CustomerNo"

# 2. Đọc dữ liệu gốc
# Chú ý: Nếu cột Date đang ở dạng chuỗi "12/9/2019", spark cần hiểu nó là Date để cộng trừ ngày.
# Sử dụng to_date để ép kiểu cho an toàn. Tùy định dạng ngày/tháng hay tháng/ngày (ở đây giả sử dd/MM/yyyy hoặc MM/dd/yyyy).
df_original = spark.read.csv(CSV_INPUT, header=True, inferSchema=True)

# Ép kiểu cột Date về dạng chuẩn (nếu inferSchema chưa xử lý được định dạng có dấu '/')
# Định dạng trong dữ liệu mẫu là "12/9/2019" (có thể là d/M/yyyy hoặc M/d/yyyy)
df_original = df_original.withColumn(DATE_COL, F.to_date(F.col(DATE_COL), "M/d/yyyy"))

# 3. Tạo các bản sao
# BẢN SAO 1
df_copy_1 = df_original.withColumn(
    CUSTOMER_COL, F.concat(F.lit("COPY1_"), F.col(CUSTOMER_COL).cast("string"))
).withColumn(
    TRANSACTION_COL, F.concat(F.lit("COPY1_"), F.col(TRANSACTION_COL).cast("string"))
).withColumn(
    # Dùng Hash của mã hóa đơn để lấy dư cho 7, sau đó trừ 3 (Tạo ra số từ -3 đến +3)
    # Cách này đảm bảo các món trong cùng 1 Invoice sẽ luôn được cộng/trừ cùng 1 số ngày
    DATE_COL, F.expr(f"date_add({DATE_COL}, cast((abs(hash({TRANSACTION_COL})) % 7) - 3 as int))")
)

# BẢN SAO 2
df_copy_2 = df_original.withColumn(
    CUSTOMER_COL, F.concat(F.lit("COPY2_"), F.col(CUSTOMER_COL).cast("string"))
).withColumn(
    TRANSACTION_COL, F.concat(F.lit("COPY2_"), F.col(TRANSACTION_COL).cast("string"))
).withColumn(
    DATE_COL, F.expr(f"date_add({DATE_COL}, cast((abs(hash({TRANSACTION_COL})) % 7) - 3 as int))")
)

# 4. Gộp 3 DataFrame (Bản gốc + 2 bản sao)
df_augmented = df_original.unionAll(df_copy_1).unionAll(df_copy_2)

# Chuyển lại cột Date thành chuỗi định dạng ban đầu để xuất file (nếu muốn giữ nguyên format)
df_augmented = df_augmented.withColumn(DATE_COL, F.date_format(F.col(DATE_COL), "M/d/yyyy"))

# Gom dữ liệu về 1 phân vùng để khi xuất ra không bị chia thành nhiều file nhỏ
df_final = df_augmented.coalesce(1)

# 5. Xuất ra file CSV
print(f"--- Đang lưu file CSV tại: {OUTPUT_CSV} ---")
df_final.write.mode("overwrite").csv(OUTPUT_CSV, header=True)

# 6. Xuất ra file Parquet
print(f"--- Đang lưu file Parquet tại: {OUTPUT_PARQUET} ---")
df_final.write.mode("overwrite").parquet(OUTPUT_PARQUET)

spark.stop()
print("--- Hoàn thành quá trình xử lý và lưu file an toàn cho cả K-Means & FP-Growth! ---")

your 131072x1 screen size is bogus. expect trouble
26/04/28 06:44:52 WARN Utils: Your hostname, admin2025RESNAN resolves to a loopback address: 127.0.1.1; using 172.27.14.10 instead (on interface eth0)
26/04/28 06:44:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/28 06:44:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/28 06:44:55 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


--- Đang lưu file CSV tại: ../data_augmented.csv ---


--- Đang lưu file Parquet tại: ../data.parquet ---


--- Hoàn thành quá trình xử lý và lưu file an toàn cho cả K-Means & FP-Growth! ---
